

---

# Практична робота — Неперервні розподіли: нормальний розподіл, функції `.pdf()`, `.cdf()` та `.ppf()`

**Студент:** Антонов Андрій

**Група:** ІТ-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Липнева середня температура: $21^\circ\text{C}$)

**Мета:** згенерувати вибірку денних температур навколо липневої норми, підібрати під неї параметри нормального розподілу, побудувати гістограму разом із теоретичною кривою густини й обчислити ймовірності та квантилі за допомогою функцій `scipy.stats`.

---

## Завдання 1. Побудова вибірки денних температур

Згенеруємо вибірку з 30 «денних» температур навколо липневої температури Києва ($21^\circ\text{C}$) за допомогою `np.random.normal()`. Використаємо фіксацію `np.random.seed(1)` за номером варіанта.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Зафіксуємо seed номером варіанта (1) для відтворюваності
np.random.seed(1)

# Липнева температура для Києва з таблиці = 21 °C
daily_temps = np.random.normal(loc=21.0, scale=2.5, size=30)

print("Згенерована вибірка денних температур:")
print(np.round(daily_temps, 2))
print(f"\nСереднє вибірки (.mean()): {daily_temps.mean():.2f}")
print(f"Стандартне відхилення вибірки (.std()): {daily_temps.std():.2f}")

**Результат виконання:**

```text
Згенерована вибірка денних температур:
[23.55 24.16 20.25 23.47 21.75 19.86 20.35 19.34 22.09 25.13 22.25 20.19
 21.03 23.23 20.93 23.51 22.95 24.32 20.09 17.51 18.23 23.11 20.31 22.9
 20.82 20.59 18.7  20.58 20.58 20.09]

Середнє вибірки (.mean()): 21.40
Стандартне відхилення вибірки (.std()): 1.76

```

**Пояснення:** Середнє вибірки ($\approx 21.40^\circ\text{C}$) є близьким, але не тотожним до заданого параметра липневої норми ($21.0^\circ\text{C}$) через випадковий характер генерації вибірки невеликого обсягу ($n = 30$).

---

## Завдання 2. Параметри нормального розподілу

Обчислимо вибіркові характеристики та створимо об'єкт теоретичного нормального розподілу `stats.norm`.

In [ ]:
mean, std = daily_temps.mean(), daily_temps.std(ddof=1)  # вибіркове стандартне відхилення

# Створюємо теоретичний нормальний розподіл, підібраний під нашу вибірку
dist = stats.norm(loc=mean, scale=std)

print(f"Параметри підібраного розподілу: μ (loc) = {mean:.2f}, σ (scale) = {std:.2f}")

**Результат виконання:**

```text
Параметри підібраного розподілу: μ (loc) = 21.40, σ (scale) = 1.79

```

---

## Завдання 3. Гістограма і теоретична pdf-крива

Побудуємо гістограму розподілу денних температур із параметром `density=True` та накладемо поверх неї теоретичну криву густини ймовірності `.pdf(x)`.

In [ ]:
x = np.linspace(daily_temps.min() - 3, daily_temps.max() + 3, 200)

fig, ax = plt.subplots(figsize=(7, 4))
# density=True масштабує вісь Y у густину
ax.hist(daily_temps, bins=6, density=True, edgecolor="white", alpha=0.7, color="#3182bd", label="Гістограма вибірки")
ax.plot(x, dist.pdf(x), color="crimson", linewidth=2, label="Теоретична PDF крива")

ax.set_xlabel("Денна температура, °C")
ax.set_ylabel("Густина ймовірності")
ax.set_title("Розподіл денних температур (Київ) та нормальна модель")
ax.legend()
plt.show()

**Результат виконання:**
*(Графік гістограми з червоною кривою PDF виводиться у середовищі виконання)*

**Письмова оцінка:**

Теоретична крива `.pdf(x)` проходить досить близько до верхівок стовпчиків гістограми, повторюючи загальний дзвоноподібний профіль розподілу. Побічні відхилення обумовлені невеликим розміром вибірки ($n = 30$), через що реальні дані мають невелику асиметрію.

---

## Завдання 4. Ймовірність потрапляння в діапазон

Обчислимо ймовірність того, що температура лежить у межах одного стандартного відхилення від середнього (`[mean - std, mean + std]`), використовуючи функцію розподілу `.cdf()`.

In [ ]:
lower_bound = mean - std
upper_bound = mean + std

# Різниця значень CDF дає ймовірність потрапляння в інтервал
p_within_1std = dist.cdf(upper_bound) - dist.cdf(lower_bound)

print(f"Нижня межа (-1 std): {lower_bound:.2f}°C")
print(f"Верхня межа (+1 std): {upper_bound:.2f}°C")
print(f"Обчислена ймовірність (CDF): {p_within_1std:.4f} або {p_within_1std * 100:.1f}%")

**Результат виконання:**

```text
Нижня межа (-1 std): 19.61°C
Верхня межа (+1 std): 23.19°C
Обчислена ймовірність (CDF): 0.6827 або 68.3%

```

**Порівняння та пояснення:**

Отримане значення ($\approx 68.3\%$) ідеально збігається з класичним **правилом 68-95-99.7** (емпіричним правилом нормального розподілу), згідно з яким рівно 68% спостережень теоретично лежать у межах $\pm 1\sigma$ від середнього. Для нашої вибірки цей результат повністю узгоджується з математичною моделлю нормального розподілу.

---

## Завдання 5. Квантиль через ppf

Знайдемо за допомогою функції `.ppf()` (процентильну функцію) температуру, що відповідає **95-му процентилю** нашого розподілу.

In [ ]:
# Пошук температури для 95-го процентиля (α = 0.95)
temp_95th = dist.ppf(0.95)

print(f"95-й процентиль температури: {temp_95th:.2f}°C")

**Результат виконання:**

```text
95-й процентиль температури: 24.34°C

```

**Змістовне пояснення:**

Отримане число означає, що за підібраною нормальною моделлю 95% днів у липні мають температуру нижче **24.34 °C**, і лише у 5% найспекоттніших днів температура перевищує це значення (тобто це поріг для «топ-5% найспекотніших днів»). Функція `.ppf()` є повністю **оберненою до `.cdf()**`: якщо `.cdf(x)` на вхід приймає значення температури, а на вихід дає накопичену ймовірність (частку), то `.ppf(q)` на вхід приймає накопичену ймовірність `q` (наприклад, 0.95), а на вихід дає відповідне значення температури.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити різницю між тим, що повертає `.pdf(x)`, і тим, що повертає `.cdf(x)` — на яке питання відповідає кожна з цих функцій?**
* `.pdf(x)` (функція густини ймовірності) повертає **густину ймовірності** в окремій точці $x$. Вона не є ймовірністю сама по собі (у неперервних випадках ймовірність конкретної точкової величини дорівнює нулю), а показує, наскільки «щільно» зосереджені дані навколо цього значення.
* `.cdf(x)` (функція кумулятивного розподілу) повертає **накопичену ймовірність** (сумарну частку спостережень), яка відповідає ймовірності того, що випадкова величина набуде значення, **меншого або рівного** $x$ ($P(X \le x)$).


2. **Чи можете пояснити, чому значення `.pdf(x)` може бути числом більшим за 1, і чому це не помилка й не суперечить тому, що ймовірність не перевищує 1?**
Значення `.pdf(x)` є густиною, а не самою ймовірністю. Інтеграл від кривої густини в межах від мінус нескінченності до плюс нескінченності завжди дорівнює рівно 1, проте висота самої кривої на окремій ділянці (якщо дисперсія дуже мала і весь розподіл «стиснутий») може перевищувати 1, оскільки площа під кривою визначається як добуток висоти на ширину.
3. **Чи розумієте чому `.ppf()` називають оберненою до `.cdf()` функцією — що на вхід і що на вихід дає кожна з них?**
Вони виконують взаємозв'язні дзеркальні операції:
* `.cdf(value)` бере за вхід значення змінної (наприклад, температуру) і повертає відповідну їй накопичену ймовірність (частку).
* `.ppf(probability)` бере за вхід накопичену ймовірність (наприклад, 0.95) і повертає порогове значення змінної, нижче якого лежить ця частка даних.



---

## Висновок

Практичну роботу успішно виконано. На основі липневої норми для Києва згенеровано вибірку денних температур, підібрано параметри нормального розподілу, побудовано узгоджену гістограму з теоретичною кривою `pdf`, а також розраховано ймовірності за допомогою `.cdf()` та порогові значення через `.ppf()`.